# Introduction

In this tutorial, we will use the **UNI model** from [MahmoodLab](https://github.com/mahmoodlab/UNI) (follow the Installation steps), a self-supervised vision transformer trained on histology slides, to extract **patch-level feature embeddings** from Whole Slide Images (WSIs).

The steps we will follow are:

1. **Load** the pretrained UNI model and its checkpoint weights.
2. **Tokenize** the WSI into small, non-overlapping patches of size **224×224 pixels**.
3. **Save** each patch image and record its spatial location (X, Y coordinates) on the original WSI.
4. **Feed** the patches through the pretrained UNI model to extract deep **feature embeddings**.
5. **Link** each embedding with its spatial metadata (Slide ID, Patch ID, coordinates).
6. **Save** the final dataset for downstream spatial analysis tasks.

---

By following this process, we create a compact and meaningful representation of very large WSIs, which can be easily used for downstream tasks.


# Imports and Environment Setup

In this section, we import the necessary libraries and set up the environment.

- **PyTorch** (`torch`) and **TorchVision**: for deep learning operations and image handling.
- **Pandas**, **NumPy**: for working with tabular data and arrays.
- **PIL** (`Image`): for opening and processing images.
- **Tifffile**: for reading `.tiff` format images, commonly used in digital pathology.
- **UNI model utilities**: 
  - `get_encoder` to load the pretrained UNI model.
  - `extract_patch_features_from_dataloader` to extract embeddings from patches.
  - Various evaluation utilities for downstream analysis (e.g., linear probe, k-NN, ProtoNet).

Finally, we set the **device** to use a **GPU** if available, otherwise fallback to **CPU**:
```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [1]:
import anndata as ad
import math
import numpy as np
import os
from os.path import join as j_
import pandas as pd
from pathlib import Path
from PIL import Image
import sys
import timm
import tifffile
import torch
import torchvision
import warnings

from torchvision import transforms

sys.path.append('..')
from uni import get_encoder
from uni.downstream.extract_patch_features import extract_patch_features_from_dataloader
from uni.downstream.eval_patch_features.linear_probe import eval_linear_probe
from uni.downstream.eval_patch_features.fewshot import eval_knn, eval_fewshot
from uni.downstream.eval_patch_features.protonet import ProtoNet, prototype_topk_vote
from uni.downstream.eval_patch_features.metrics import get_eval_metrics, print_metrics
from uni.downstream.utils import concat_images

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/charlottewolf/miniconda3/envs/autoencoder/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Stain normalisation - Preprocessing

# Loading the Pretrained UNI Model and Preparing Image Transformations

In this section, we load the **pretrained UNI model** and prepare the **transformations** that will be applied to the image patches before feeding them into the model.

Steps:

1. **Import additional libraries**:
   - `timm`: a popular library that provides implementations of vision transformer models.
   - `huggingface_hub`: for downloading model weights (optional here if already downloaded).

2. **Set the local directory**:
   - `local_dir` is where the pretrained model checkpoint (`pytorch_model.bin`) is stored. Here 'checkpoint_path' is an exampl, set where your checkpoint is located.
   - (Optional) If not downloaded yet, you could use `hf_hub_download` to fetch the weights from Hugging Face.

3. **Model setup**:
   - We define the model architecture using `timm_kwargs` to match the configuration used by UNI (based on a Vision Transformer (ViT) backbone).
   - The model is created with `timm.create_model(**timm_kwargs)`.
   - Pretrained weights are loaded with `model.load_state_dict(...)`.
   - The model is set to evaluation mode (`model.eval()`) and moved to the selected device (CPU or GPU).

4. **Image preprocessing**:
   - Before passing patches to the model, we apply a series of transformations:
     - **Resize** the patch to 224×224 pixels.
     - **Convert** the image to a PyTorch tensor.
     - **Normalize** pixel values using ImageNet statistics (mean and std).



In [2]:
#from huggingface_hub import login, hf_hub_download

#login(token="xxx")  
pretrained = True

weights_dir = '../weights/'
Path(weights_dir).mkdir(exist_ok=True)  
#hf_hub_download("MahmoodLab/UNI2-h", filename="pytorch_model.bin", local_dir=local_dir, force_download=True)
timm_kwargs = {'model_name': 'vit_giant_patch14_224',
    'img_size': 224, 
               'patch_size': 14, 
               'depth': 24,
               'num_heads': 24,
               'init_values': 1e-5, 
               'embed_dim': 1536,
               'mlp_ratio': 2.66667*2,
               'num_classes': 0, 
               'no_embed_class': True,
               'mlp_layer': timm.layers.SwiGLUPacked, 
               'act_layer': torch.nn.SiLU, 
               'reg_tokens': 8, 
               'dynamic_img_size': True
              }
model = timm.create_model(**timm_kwargs)
model.load_state_dict(torch.load(os.path.join(weights_dir, "pytorch_model.bin"), map_location="cpu"), strict=True)
model.eval()
model.to(device)
transform = transforms.Compose(
    [
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)


/tmp/ipykernel_445094/317336369.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(os.path.join(weights_dir, "pytorch_model.bin"), map_loc

# Tokenizing Whole Slide Images (WSIs) into Patches

In this section, we split each Whole Slide Image (WSI) into small non-overlapping patches and save them as individual image files, along with recording their spatial metadata.

Steps:

1. **Set variables**:
   - `sample`: data is differntiated through sample IDs. From 10x: Colorectal Cancer: **CRC_P1, CRC_P2, CRC_P5**. Non-cancerous: **Normal_P3, Normal_P5**. **Combined**: refers to a combined folder that contains information for ALL samples.  
   - `bin_size`: Visium HD bin size. Choose between 002, 008, 016 µm. 
   - `patch_size`: `224 × 224` pixels.
   - `stride`: `224` pixels (non-overlapping patches).

2. a) **Filenames**  
   Note: as a convention all filenames/filepaths start with the prefix "`f_`". If a distinction between datatypes is needed, the datatype will be added at the end, as seen in the following two filenames.  
   - `f_metadata_csv`: filename for csv file that will be generated during patch creation. Stores information such as paths to patches and patch names. 
   - `f_metadata_h5ad`: filename for h5ad file that will be generated during embedding creation. Stores information such as patch ID and embedding per patch. 
   - `f_wsi`: filename for original WSI

2. b) **Set up paths**:  
   Note: as a convention all paths to directories other than the base directory start with the prefix "`dir_`".  
   - `basedir`: relative path to basic data directory. Note that none of the following paths have to be edited in any way if you follow the same folder structure. 
   - `dir_sample`: Path to data directory for specific sample.
   - `dir_wsi`: Path to WSI. 
   - `dir_base_output`: Path to all transcriptomics information.
   - `dir_histology`: Path to output directory for any unimodel results. 
   - `dir_metadata`: Subdirectory of resultpath. Stores all metadata.
   - `dir_patches`: Template path to directory containing patches. 

3. **Retreiving the frame coordinates**:
   - 10x genomics/the spaceranger pipeline provide a `tissuepositions.parquet` file, which notes the pixel postion for each bin. 
   From here, we extract the top left and bottom right pixel corner of the image. We use this later to crop only the necessary part from the `.btf`-file, since it is very large and captures the entire tissue slide, not just our desired frame. 

4. **Processing the WSI**:
   - Load the `.btf` file using `tifffile.imread()`.
   - Convert the numpy array to a PIL image for easier patch extraction.
   - Loop over the WSI grid and **crop** patches at regular intervals.
   - Save each patch as a `.png` image.
   - Record metadata for each patch:
     - Slide ID (origin WSI)
     - Patch filename
     - Top-left (X, Y) coordinate of the patch within the slide
     - Path where the patch is stored

5. **Handling errors**:
   - If a WSI fails to load or process, it is added to `failed_wsi_files`.

6. **Metadata saving**:
   - After processing all WSIs, the collected metadata is stored in a single CSV file (`WSI_patch_metadata_prostate.csv`).

---

The metadata will also be important for linking back embeddings to their original spatial locations on the WSI.


In the following, please choose the sample (at the time of writing: either CRC_Px or Normal_Px) and the bin size (2um, 8um or 16um).

In [4]:
# Basic Variables
sample = "CRC_P11mm" # Adapt as needed. Please note: sample cannot be "combined" for patch creation, only for creating embeddings.
bin_size = "002"  # Adapt as needed

patch_size = (224, 224) # determines patch size in pixels 
stride = 224

if sample == "combined":
    warnings.warn("Sample is set as combined. This can only be used in case of embeddings, not during patch creation.")
    print("Continue...")


# Filenames
f_metadata_csv = "WSI_patch_metadata.csv"   
f_metadata_h5ad = "WSI_patch_metadata.h5ad"  
f_wsi = "Visium_HD_6p5mm_Human_Colon_Cancer_tissue_image.btf" # Adapt as needed


# Paths
basedir = Path("../../../../data") # base directory containing all data
dir_sample = basedir / sample # Path to data directory of specific sample 
if not dir_sample.is_dir():
    raise NotADirectoryError(f"Path {dir_sample} does not exist.")
dir_wsi = dir_sample / "input" / f_wsi  # Path to high-res WSI
dir_base_output = dir_sample / f"output/binned_outputs/square_{bin_size}um"

dir_histology = dir_sample / "01_preprocessing/unimodel" # Path to any ouputs created by unimodel
dir_histology.mkdir(parents=True, exist_ok=True)

dir_metadata = dir_histology / "metadata" # Path to metadata directory (contains metadata for unimodel outputs)
dir_metadata.mkdir(exist_ok=True)

dir_patches = dir_histology / "patches" # Path to store patches
dir_patches.mkdir(exist_ok=True)

dir_combined = basedir / "combined/01_preprocessing/unimodel/metadata"
dir_combined.mkdir(exist_ok=True)

Get the minimum pixel coordinates for the top left pixel corner and the maximum pixel coordinates for the bottom right corner of the desired frame/region to crop. 

In [3]:
# Get crop cordinates for spatial transcirptomics area from the "tissue_positions.parquet" file
col = "pxl_col_in_fullres"
row = "pxl_row_in_fullres" 

if sample == "combined":
    raise NameError("Directory 'combined' cannot be used during patch creation, only embedding creation. Please change the variable 'sample' to a working directory.")


# Get crop cordinates for spatial transcirptomics area from the "tissue_positions.parquet" file
tissuepos = pd.read_parquet(dir_base_output / "spatial/tissue_positions.parquet")

min_coord = [tissuepos.loc[tissuepos[col] > 0, col].min(), tissuepos.loc[tissuepos[row] > 0, row].min()]
max_coord = [tissuepos.loc[tissuepos[col] > 0, col].max(), tissuepos.loc[tissuepos[row] > 0, row].max()]
print(f"Minimum coordinates: {min_coord} (row, column)")
print(f"Maximum coordinates: {max_coord} (row, column)")

# Coordinates for cropping (top-left corner)
#x0, y0 = math.floor(min_coordinates[0]), math.floor(min_coordinates[1])
#width, height = math.ceil(max_coordinates[0]) - x0, math.ceil(max_coordinates[1]) - y0
xmin, ymin = math.floor(min_coord[0]), math.floor(min_coord[1])
xmax, ymax = math.ceil(max_coord[0]), math.ceil(max_coord[1])

Minimum coordinates: [np.float64(4582.488973049297), np.float64(4813.179480034815)] (row, column)
Maximum coordinates: [np.float64(30120.701648030503), np.float64(30351.049063254224)] (row, column)


1. Load the image partially
2. Crop out desired region
3. Optionally save to .tif file
4. Convert to numpy array

In [4]:
import pyvips

# Open the image (does not load it fully)
image = pyvips.Image.new_from_file(dir_wsi, access="sequential")

cropped = image.crop(xmin, ymin, xmax - xmin, ymax - ymin)

# Convert to NumPy or Save to disk or (if needed)

# Uncomment to save the cropped region to a file:
# wsi_image.write_to_file(os.path.join(basepath, "input/Cropped_Region.tif"))

# Convert to NumPy array
numpy_array = np.ndarray(
    buffer=cropped.write_to_memory(),
    dtype=np.uint8,
    shape=[cropped.height, cropped.width, cropped.bands],
)

# Convert to PIL image
wsi_image = Image.fromarray(numpy_array)

In [ ]:
failed_wsi_files = []
all_patch_metadata = [] 

try:
    wsi_width = xmax - xmin
    wsi_height = ymax - ymin

    print(f"Processing {sample} - Size: {wsi_width}x{wsi_height}")

    saved_patches = 0
    for x in range(0, wsi_width - patch_size[0], stride):
        for y in range(0, wsi_height - patch_size[1], stride):
            patch = wsi_image.crop((x, y, x + patch_size[0], y + patch_size[1]))
        
            f_patch = f"patch_{saved_patches+1}.png"
            dir_patch = dir_patches / f_patch
            patch.save(dir_patch)

            
            # all_patch_metadata.append([slide_name, patch_filename, x, y, patch_path])
            abs_x = x + xmin
            abs_y = y + ymin
            all_patch_metadata.append([sample, f_patch, abs_x, abs_y, dir_patch])
            saved_patches += 1

    print(f"Saved {saved_patches} patches for {sample}")

except Exception as e:
    print(f"Error processing {f_wsi}: {e}")
    failed_wsi_files.append(f_wsi)


metadata_df = pd.DataFrame(all_patch_metadata, columns=['Slide_ID', 'Patch_ID', 'X', 'Y', 'Path'])

def save_or_append_metadata(df: pd.DataFrame, target_path: Path):
    target_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure folder exists
    
    if target_path.exists():
        existing_df = pd.read_csv(target_path)
        combined_df = pd.concat([existing_df, df], ignore_index=True)
    else:
        combined_df = df
        
    combined_df.to_csv(target_path, index=False)
    print(f"Metadata saved/updated at: {target_path}")

# 1. Save / Append to local sample directory
dir_csv_metadata = dir_metadata / f_metadata_csv
save_or_append_metadata(metadata_df, dir_csv_metadata)

# 2. Save / Append to combined directory
dir_combined_csv = dir_combined / f_metadata_csv
save_or_append_metadata(metadata_df, dir_combined_csv)

print("All WSIs processed successfully!")
print("Failed WSIs:", failed_wsi_files)
print(f"Single metadata saved in {dir_csv_metadata}")


Processing CRC_P6_5mm - Size: 25539x25539
Saved 12996 patches for CRC_P6_5mm
All WSIs processed successfully!
Failed WSIs: []
Metadata saved in ../../../../data/CRC_P6_5mm/01_preprocessing/unimodel/metadata/WSI_patch_metadata.csv


# Extracting Patch Feature Embeddings Using the UNI Model

In this section, we load the extracted patches, pass them through the pretrained UNI model, and save their feature embeddings along with spatial metadata.

Steps:

1. **Patch Dataset Preparation**:
   - Define a `PatchDataset` class that:
     - Loads all `.png` patch images from a given directory.
     - Applies the same image transformations we defined earlier.
     - Returns an image and a dummy label (not used here).

2. **Loading Patches and Creating DataLoader**:
   - For each WSI (each directory of patches):
     - Create a `PatchDataset` and corresponding `DataLoader`.
     - Load patches in small batches (batch size = 4) without shuffling (preserve patch order).
     - **Note**: For faster parallel processing, adapt `num_workers = 12` to a number of cores that works for your system. 

3. **Extract Patch Features**:
   - Use the `extract_patch_features_from_dataloader` function from UNI to pass the patches through the model.
   - Extract the feature embeddings for each patch.

4. **Collect Embeddings and Metadata**:
   - For each patch, store:
     - Its extracted embedding.
     - The corresponding Slide_ID (WSI name).
     - The Patch_ID (patch filename).

5. **Merge with Patch Location Metadata**:
   - Combine the extracted embeddings with the previously saved patch spatial information (`X`, `Y` coordinates) using a `match_id` (Slide_ID + Patch_ID).

6. **Save Final Data**:
   - Store everything in an Anndata object file `WSI_patch_embeddings_prostate.h5ad`, which contains:  
     AnnData object with n_obs × n_vars = n_patches × embedding_dim  
      ├── `.X`               → patch embeddings (from UNI model)  
      ├── `.obs`             → Slide_ID, Patch_ID, X, Y  
      ├── `.obsm["spatial"]` → [X, Y] coordinates for spatial plotting  
      └── `.uns`             → currently empty 
   - Optinal: save as .csv (currently out-commmented)

---

At the end of this step, we will have a **structured table** linking each patch's location to its **deep feature representation**, ready for downstream spatial analysis tasks.


In [ ]:
import gc
import glob
import math
import os
import random
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

basedir = Path("../../../../data")
skip_samples = ["new", "CRC_P1", "CRC_P2", "CRC_P5", "Normal_P3", "Normal_P5"]

f_metadata_h5ad = "WSI_patch_metadata.h5ad"  
f_metadata_csv = "WSI_patch_metadata.csv"  # Name of your master CSV file

# --- CHANGE 1: Point to the Master CSV file instead of H5AD ---
master_csv_path = basedir / "combined/01_preprocessing/unimodel/metadata" / f_metadata_csv

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Global seed set to: {seed}")

set_seed(42)

def check_for_patches(basedir, skip_samples=None):
    if skip_samples is None:
        skip_samples = ["CRC_P3", "Normal_P1"]

    basedir_path = Path(basedir)
    sample_dirs = [
        p for p in basedir_path.iterdir() 
        if p.is_dir() and p.name != "combined" and p.name not in skip_samples
    ]
    
    valid_patch_dirs = []
    for sample_dir in sample_dirs:
        patch_dir = sample_dir / "01_preprocessing" / "unimodel" / "patches"
        if patch_dir.is_dir() and any(patch_dir.glob("*.png")):
            valid_patch_dirs.append(patch_dir)
        else:
            print(f"Skipping {sample_dir.name}: No patch PNGs found at {patch_dir}")
            
    if not valid_patch_dirs:
        raise FileNotFoundError(f"No valid sample patch directories found inside {basedir}")

    print(f"Found {len(valid_patch_dirs)} valid sample patch directories: ", valid_patch_dirs)
    return valid_patch_dirs

# ============================================================
# 1. Dataset & Setup
# ============================================================
class PatchDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = sorted(glob.glob(os.path.join(root_dir, "*.png")))
        self.transform = transform
        self.patch_ids = [os.path.basename(img) for img in self.image_paths]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, -1

num_prefix_tokens = getattr(model, "num_prefix_tokens", 1 + timm_kwargs.get("reg_tokens", 0))
grid_h, grid_w = model.patch_embed.grid_size  # (16, 16)
n_tokens = grid_h * grid_w
token_patch_px = timm_kwargs["patch_size"]    # 14

# ============================================================
# 2. Optimized Forward Pass
# ============================================================
@torch.no_grad()
def extract_patch_and_token_features(model, dataloader, device):
    all_patch_embeddings = []
    all_token_embeddings = []
    
    for batch, _ in dataloader:
        batch = batch.to(device)
        feats = model.forward_features(batch)
        
        patch_emb = model.forward_head(feats, pre_logits=True)
        all_patch_embeddings.append(patch_emb.cpu())
        
        patch_tokens = feats[:, num_prefix_tokens:, :].to(torch.float16)
        all_token_embeddings.append(patch_tokens.cpu())

    patch_feats_tensor = torch.cat(all_patch_embeddings, dim=0)
    token_feats_tensor = torch.cat(all_token_embeddings, dim=0)
    
    return patch_feats_tensor, token_feats_tensor

# ============================================================
# 3. Main Processing Loop
# ============================================================
patch_directories = check_for_patches(basedir, skip_samples)

# --- CHANGE 2: Read CSV file and clean data types ---
if not master_csv_path.exists():
    raise FileNotFoundError(f"Master metadata CSV file not found at {master_csv_path}")

print(f"Loading master spatial metadata from {master_csv_path}...")
master_metadata_df = pd.read_csv(master_csv_path)

# Ensure string and numeric datatypes match expectations
master_metadata_df['Slide_ID'] = master_metadata_df['Slide_ID'].astype(str)
master_metadata_df['Patch_ID'] = master_metadata_df['Patch_ID'].astype(str)
master_metadata_df['X'] = master_metadata_df['X'].astype(float)
master_metadata_df['Y'] = master_metadata_df['Y'].astype(float)

master_metadata_df['match_id'] = master_metadata_df['Slide_ID'] + '_' + master_metadata_df['Patch_ID']

dir_combined_meta = basedir / "combined" / "01_preprocessing" / "unimodel" / "metadata"
dir_combined_meta.mkdir(parents=True, exist_ok=True)

combined_patch_h5ad_path = dir_combined_meta / f_metadata_h5ad
combined_token_h5ad_path = dir_combined_meta / "WSI_patch_token_embeddings.h5ad"

all_patch_adata_list = []

for dir_path in patch_directories:
    sample_name = dir_path.parents[2].name
    print(f"\n================ Processing Sample: {sample_name} ================")

    metadata_dir = dir_path.parent / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)

    # Filter master metadata specifically for current sample
    sample_metadata_df = master_metadata_df[master_metadata_df['Slide_ID'] == sample_name].copy()
    if sample_metadata_df.empty:
        print(f"Skipping {sample_name}: No matching metadata found in master CSV.")
        continue

    test_dataset = PatchDataset(root_dir=dir_path, transform=transform)
    test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=12)

    # A. Feature extraction
    patch_feats, token_feats = extract_patch_and_token_features(model, test_dataloader, device)

    # --------------------------------------------------------
    # PART I: Patch-Level AnnData (224x224)
    # --------------------------------------------------------
    df_patch = pd.DataFrame(patch_feats.numpy())
    df_patch["Slide_ID"] = sample_name
    df_patch["Patch_ID"] = test_dataset.patch_ids
    df_patch['match_id'] = df_patch['Slide_ID'].astype(str) + '_' + df_patch['Patch_ID'].astype(str)
    
    init_len = len(df_patch)
    df_patch = df_patch.merge(sample_metadata_df[['match_id', 'X', 'Y']], on='match_id', how='left')
    if len(df_patch) != init_len:
        print(f"Warning: Row count mismatch in patch merge ({init_len} -> {len(df_patch)})!")

    meta_cols = ["Slide_ID", "Patch_ID", "X", "Y"]
    feat_cols = [c for c in df_patch.columns if c not in meta_cols + ["match_id"]]

    adata_patch = ad.AnnData(X=df_patch[feat_cols].values, obs=df_patch[meta_cols].copy())
    adata_patch.obsm["spatial"] = df_patch[["X", "Y"]].values
    adata_patch.obs.index = df_patch["Slide_ID"].astype(str) + "_" + df_patch["Patch_ID"].astype(str)

    # Save per-sample patch .h5ad
    patch_h5ad_path = metadata_dir / f_metadata_h5ad
    adata_patch.write(patch_h5ad_path)
    print(f" Saved Patch AnnData: {patch_h5ad_path}")
    all_patch_adata_list.append(adata_patch)

    # --------------------------------------------------------
    # PART II: Token-Level AnnData (14x14 Sub-patches)
    # --------------------------------------------------------
    token_matrix = token_feats.reshape(-1, token_feats.shape[-1]).numpy()
    
    t_sample_ids, t_patch_ids, t_ids, t_rows, t_cols = [], [], [], [], []
    for p_id in test_dataset.patch_ids:
        for t in range(n_tokens):
            t_sample_ids.append(sample_name)
            t_patch_ids.append(p_id)
            t_ids.append(t)
            t_rows.append(t // grid_w)
            t_cols.append(t % grid_w)

    obs_tokens = pd.DataFrame({
        "Slide_ID": t_sample_ids,
        "Patch_ID": t_patch_ids,
        "Token_ID": np.array(t_ids, dtype=np.uint16),
        "Token_row": np.array(t_rows, dtype=np.uint8),
        "Token_col": np.array(t_cols, dtype=np.uint8),
    })

    obs_tokens["match_id"] = obs_tokens["Slide_ID"].astype(str) + "_" + obs_tokens["Patch_ID"].astype(str)
    obs_tokens = obs_tokens.merge(sample_metadata_df[["match_id", "X", "Y"]], on="match_id", how="left").drop(columns="match_id")

    obs_tokens["Token_X"] = obs_tokens["X"] + obs_tokens["Token_col"] * token_patch_px
    obs_tokens["Token_Y"] = obs_tokens["Y"] + obs_tokens["Token_row"] * token_patch_px
    obs_tokens.index = obs_tokens["Slide_ID"].astype(str) + "_" + obs_tokens["Patch_ID"].astype(str) + "_token" + obs_tokens["Token_ID"].astype(str)

    adata_token = ad.AnnData(X=token_matrix, obs=obs_tokens)
    adata_token.obsm["spatial"] = obs_tokens[["Token_X", "Token_Y"]].values

    # 1. Save per-sample token .h5ad
    token_h5ad_path = metadata_dir / "WSI_patch_token_embeddings.h5ad"
    adata_token.write(token_h5ad_path, compression="gzip")
    print(f" Saved Sample Token AnnData: {token_h5ad_path}")

    # 2. Append directly to master combined token .h5ad on disk
    if combined_token_h5ad_path.exists():
        existing_tokens = ad.read_h5ad(combined_token_h5ad_path)
        combined_tokens = ad.concat([existing_tokens, adata_token], join="outer", index_unique=None)
        combined_tokens.write(combined_token_h5ad_path, compression="gzip")
        del existing_tokens, combined_tokens
    else:
        adata_token.write(combined_token_h5ad_path, compression="gzip")
    print(f" Updated Combined Token AnnData on disk.")

    # 3. Memory cleanup
    del patch_feats, token_feats, token_matrix, df_patch, obs_tokens, adata_patch, adata_token
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# 4. Save Master Combined Patch File
# ============================================================
if all_patch_adata_list:
    print("\nWriting master combined Patch AnnData...")
    new_patch_adata = ad.concat(all_patch_adata_list, join='outer', index_unique=None)
    
    if combined_patch_h5ad_path.exists():
        existing_patch_adata = ad.read_h5ad(combined_patch_h5ad_path)
        combined_patch_adata = ad.concat([existing_patch_adata, new_patch_adata], join='outer', index_unique=None)
        combined_patch_adata.write(combined_patch_h5ad_path)
    else:
        new_patch_adata.write(combined_patch_h5ad_path)

print("\nPipeline complete!")